# From REINFORCE to PPO: Policy Gradient Methods

**Last lecture:** Q-learning and DQN break on continuous actions. We need a method that *directly outputs* continuous torques.

**This lecture:** We train a policy network $\pi_\theta(s) \to a \in \mathbb{R}^n$ using gradient ascent on expected reward.

| Demo | Algorithm | What it shows |
|------|-----------|---------------|
| 1 | REINFORCE | Policy gradients work! But noisy and slow. |
| 2 | PPO | Trust regions + clipping → stable, fast training. Solves the crawler with continuous actions. |
| 3 | PPO scales | Same algorithm on Ant (8 legs, 111D state, 8D action). No fundamental changes. |
| 4 | Reward engineering | Change the reward → change the behavior. Same algorithm, same environment. |

In [ ]:
# Setup
!pip install -q mujoco gymnasium stable-baselines3 shimmy

import numpy as np
import mujoco
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

print(f'MuJoCo: {mujoco.__version__}')
print(f'Gymnasium: {gym.__version__}')
print('Setup complete!')

In [ ]:
# ============================================================
# Same 2D Crawler from Lecture 1, now as a Gymnasium env
# with CONTINUOUS actions
# ============================================================

CRAWLER_XML = """
<mujoco model="crawler2d">
  <compiler angle="degree" inertiafromgeom="true"/>
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4"/>

  <default>
    <geom conaffinity="1" condim="3" friction="1.5 0.5 0.1" density="1000"/>
    <joint armature="0.1" damping="0.5"/>
  </default>

  <asset>
    <texture type="2d" name="grid" builtin="checker" width="512" height="512"
             rgb1="0.7 0.9 0.7" rgb2="0.6 0.85 0.6"/>
    <material name="grid" texture="grid" texrepeat="8 8"/>
  </asset>

  <worldbody>
    <light diffuse="0.8 0.8 0.8" pos="0 -2 3" dir="0 0.5 -1"/>
    <geom name="floor" type="plane" size="50 1 0.1" material="grid"/>

    <camera name="side" pos="0 -0.8 0.25" xyaxes="1 0 0 0 0.3 1" mode="trackcom"/>

    <body name="torso" pos="0 0 0.06">
      <joint name="root_x" type="slide" axis="1 0 0"/>
      <joint name="root_z" type="slide" axis="0 0 1"/>
      <joint name="root_rot" type="hinge" axis="0 1 0"/>

      <geom name="torso_geom" type="capsule" size="0.05"
            fromto="-0.1 0 0 0.1 0 0" rgba="0.3 0.7 0.3 1"/>

      <body name="arm" pos="0.1 0 0">
        <joint name="arm_joint" type="hinge" axis="0 1 0"
               range="-70 70" limited="true"/>
        <geom name="arm_geom" type="capsule" size="0.04"
              fromto="0 0 0 0.12 0 0" rgba="0.9 0.6 0.2 1"/>

        <body name="hand" pos="0.12 0 0">
          <joint name="hand_joint" type="hinge" axis="0 1 0"
                 range="-70 70" limited="true"/>
          <geom name="hand_geom" type="capsule" size="0.03"
                fromto="0 0 0 0.08 0 0" rgba="0.9 0.3 0.3 1"/>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor name="arm_motor" joint="arm_joint" ctrllimited="true"
           ctrlrange="-1 1" gear="5"/>
    <motor name="hand_motor" joint="hand_joint" ctrllimited="true"
           ctrlrange="-1 1" gear="3"/>
  </actuator>
</mujoco>
"""


class CrawlerGymEnv(gym.Env):
    """Gymnasium-compatible wrapper for the 2D crawler.
    
    Continuous state (4D): arm angle, hand angle, arm vel, hand vel
    Continuous action (2D): arm torque, hand torque in [-1, 1]
    Reward: forward velocity (optionally + energy penalty)
    """
    metadata = {'render_modes': ['rgb_array'], 'render_fps': 30}

    def __init__(self, render_mode='rgb_array', max_steps=500, frame_skip=4,
                 energy_penalty=0.0, ctrl_cost=0.0):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(CRAWLER_XML)
        self.data = mujoco.MjData(self.model)
        self.max_steps = max_steps
        self.frame_skip = frame_skip
        self.energy_penalty = energy_penalty
        self.ctrl_cost = ctrl_cost
        self.steps = 0
        self.render_mode = render_mode
        self.renderer = None

        # Observation: arm_angle, hand_angle, arm_vel, hand_vel
        obs_high = np.array([np.pi, np.pi, 10.0, 10.0], dtype=np.float32)
        self.observation_space = gym.spaces.Box(-obs_high, obs_high)
        # Action: torques in [-1, 1]
        self.action_space = gym.spaces.Box(
            low=-np.ones(2, dtype=np.float32),
            high=np.ones(2, dtype=np.float32)
        )

    def _get_obs(self):
        return np.array([
            self.data.qpos[3], self.data.qpos[4],
            self.data.qvel[3], self.data.qvel[4]
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[3] = self.np_random.uniform(-0.1, 0.1)
        self.data.qpos[4] = self.np_random.uniform(-0.1, 0.1)
        mujoco.mj_forward(self.model, self.data)
        self.steps = 0
        return self._get_obs(), {}

    def step(self, action):
        x_before = self.data.qpos[0]
        self.data.ctrl[:] = np.clip(action, -1, 1)
        for _ in range(self.frame_skip):
            mujoco.mj_step(self.model, self.data)
        x_after = self.data.qpos[0]
        dt = self.frame_skip * self.model.opt.timestep

        forward_vel = (x_after - x_before) / dt
        reward = forward_vel

        # Optional penalties
        if self.ctrl_cost > 0:
            reward -= self.ctrl_cost * np.sum(action ** 2)
        if self.energy_penalty > 0:
            power = np.abs(action * np.array([self.data.qvel[3], self.data.qvel[4]]))
            reward -= self.energy_penalty * np.sum(power)

        self.steps += 1
        truncated = self.steps >= self.max_steps
        return self._get_obs(), reward, False, truncated, {'x': x_after}

    def render(self):
        if self.renderer is None:
            self.renderer = mujoco.Renderer(self.model, height=320, width=560)
        self.renderer.update_scene(self.data, camera='side')
        return self.renderer.render().copy()

    def close(self):
        if self.renderer is not None:
            self.renderer.close()
            self.renderer = None


# Quick test
env = CrawlerGymEnv()
obs, _ = env.reset()
print(f'Observation shape: {obs.shape}  (arm_angle, hand_angle, arm_vel, hand_vel)')
print(f'Action shape: {env.action_space.shape}  (arm_torque, hand_torque)')

frame = env.render()
plt.figure(figsize=(8, 4))
plt.imshow(frame)
plt.axis('off')
plt.title('Same crawler — now with continuous actions!')
plt.show()
env.close()

In [ ]:
# ============================================================
# Visualization helpers
# ============================================================

def show_video(frames, fps=30, title=None):
    """Display frames as inline video."""
    fig, ax = plt.subplots(figsize=(8, 4))
    if title:
        ax.set_title(title, fontsize=14)
    ax.axis('off')
    im = ax.imshow(frames[0])
    def update(i):
        im.set_data(frames[i])
        return [im]
    anim = animation.FuncAnimation(fig, update, frames=len(frames),
                                   interval=1000/fps, blit=True)
    plt.close()
    return HTML(anim.to_html5_video())


def collect_frames(env, policy_fn, n_steps=300):
    """Roll out policy and collect render frames."""
    frames = []
    obs, _ = env.reset()
    for _ in range(n_steps):
        frames.append(env.render())
        action = policy_fn(obs)
        obs, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    return frames


def collect_frames_sb3(env, model, n_steps=300):
    """Roll out a stable-baselines3 model."""
    frames = []
    obs, _ = env.reset()
    for _ in range(n_steps):
        frames.append(env.render())
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    return frames


def plot_rewards(rewards, window=50, title='', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(rewards, alpha=0.3, color='steelblue')
    if len(rewards) >= window:
        sm = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), sm, color='steelblue', lw=2)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    return ax

---
## Demo 1: REINFORCE — policy gradient from scratch

The simplest policy gradient algorithm:
1. Run the policy, collect a trajectory $(s_0, a_0, r_0, s_1, a_1, r_1, \ldots)$
2. Compute return $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$
3. Update: $\theta \leftarrow \theta + \alpha \sum_t G_t \nabla_\theta \log \pi_\theta(a_t | s_t)$

The key insight: **increase probability of actions that led to high returns.**

The policy network directly outputs a mean and std for each torque — a Gaussian distribution over continuous actions. No discretization needed!

In [ ]:
# ============================================================
# REINFORCE (vanilla policy gradient)
# ============================================================

class PolicyNetwork(nn.Module):
    """Gaussian policy: outputs mean, learns a separate log_std."""
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, act_dim)
        )
        self.log_std = nn.Parameter(torch.zeros(act_dim))  # learnable

    def forward(self, obs):
        mean = self.net(obs)
        std = self.log_std.exp()
        return Normal(mean, std)

    def act(self, obs_np):
        """Select action for environment interaction."""
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs_np).unsqueeze(0)
            dist = self.forward(obs_t)
            action = dist.sample()
        return action.squeeze(0).numpy()


def train_reinforce(env, n_episodes=500, gamma=0.99, lr=3e-3, verbose=True):
    """Train REINFORCE with baseline subtraction."""
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    policy = PolicyNetwork(obs_dim, act_dim)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    rewards_history = []
    baseline = 0  # running mean of returns

    t0 = time.time()
    for ep in range(n_episodes):
        obs, _ = env.reset()
        log_probs = []
        rewards_ep = []

        # Collect trajectory
        while True:
            obs_t = torch.FloatTensor(obs).unsqueeze(0)
            dist = policy(obs_t)
            action = dist.sample()
            log_prob = dist.log_prob(action).sum(dim=-1)

            obs, reward, terminated, truncated, _ = env.step(
                action.squeeze(0).detach().numpy()
            )
            log_probs.append(log_prob)
            rewards_ep.append(reward)

            if terminated or truncated:
                break

        # Compute discounted returns
        returns = []
        G = 0
        for r in reversed(rewards_ep):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)

        # Baseline subtraction (reduces variance)
        baseline = 0.9 * baseline + 0.1 * returns.mean().item()
        advantages = returns - baseline

        # Policy gradient
        log_probs_t = torch.stack(log_probs).squeeze()
        loss = -(log_probs_t * advantages).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_reward = sum(rewards_ep)
        rewards_history.append(total_reward)

        if verbose and (ep + 1) % 100 == 0:
            avg = np.mean(rewards_history[-50:])
            std_action = policy.log_std.exp().detach().numpy()
            print(f'  Episode {ep+1:4d} | Avg reward: {avg:7.1f} | '
                  f'Action std: [{std_action[0]:.2f}, {std_action[1]:.2f}]')

    elapsed = time.time() - t0
    if verbose:
        print(f'  Training completed in {elapsed:.1f}s')

    return policy, rewards_history

In [ ]:
# ---------- Demo 1: REINFORCE on the crawler ----------
print('=== Demo 1: REINFORCE with continuous actions ===')
print('Policy: obs (4D) → Gaussian(mean, std) → torques (2D)\n')

env_reinforce = CrawlerGymEnv(max_steps=300)

policy_reinforce, rewards_reinforce = train_reinforce(
    env_reinforce, n_episodes=500, lr=3e-3
)

In [ ]:
# Visualize REINFORCE training
fig, ax = plt.subplots(figsize=(10, 4))
plot_rewards(rewards_reinforce, title='REINFORCE Training Curve', ax=ax)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('Notice how noisy the training is — this is the high variance')
print('of REINFORCE. The gradient estimate uses full trajectories,')
print('so lucky/unlucky episodes create big swings.')

In [ ]:
# Watch REINFORCE policy
frames_reinforce = collect_frames(
    env_reinforce, lambda obs: policy_reinforce.act(obs), n_steps=200
)
print(f'Crawler traveled to x = {env_reinforce.data.qpos[0]:.3f}')
show_video(frames_reinforce, title='REINFORCE: continuous actions (works, but noisy training)')

### REINFORCE works, but...

- **High variance** — training curve is very noisy
- **Slow convergence** — needs many episodes
- **Fragile** — a bad update can ruin the policy, and there's no mechanism to prevent large steps

This motivates the key ideas behind PPO:
1. **Baseline / critic** → reduce variance (actor-critic)
2. **Trust region** → prevent catastrophically large updates (TRPO idea)
3. **Clipped surrogate** → simple approximation of trust region (PPO)
4. **GAE** → better advantage estimation (bias-variance tradeoff)

Rather than implementing all of these from scratch, let's use **stable-baselines3** to see the result.

---
## Demo 2: PPO solves the crawler with continuous actions

PPO = REINFORCE + critic baseline + clipped surrogate loss + GAE.

All the ideas from the lecture, packaged into one practical algorithm.

Same crawler, same continuous action space that broke DQN.

In [ ]:
# ---------- Demo 2: PPO on the crawler ----------
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback


class RewardLogger(BaseCallback):
    """Log episode rewards during training."""
    def __init__(self):
        super().__init__()
        self.episode_rewards = []

    def _on_step(self):
        # Check if any episodes finished
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
        return True


print('=== Demo 2: PPO on the crawler ===')
print('Same continuous action space that broke DQN.\n')

# Wrap the env for SB3 (it needs Monitor wrapper for episode stats)
from stable_baselines3.common.monitor import Monitor
env_ppo = Monitor(CrawlerGymEnv(max_steps=500))

logger = RewardLogger()

t0 = time.time()
ppo_model = PPO(
    'MlpPolicy', env_ppo,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,  # <-- GAE!
    clip_range=0.2,   # <-- PPO clipping!
    verbose=1
)

ppo_model.learn(total_timesteps=100_000, callback=logger)
print(f'\nTraining completed in {time.time()-t0:.0f}s')

In [ ]:
# Compare REINFORCE vs PPO training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_rewards(rewards_reinforce, title='REINFORCE (noisy, slow)', ax=axes[0])
if logger.episode_rewards:
    plot_rewards(logger.episode_rewards, title='PPO (stable, fast)', ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Watch PPO policy
env_show = CrawlerGymEnv(max_steps=300)
frames_ppo = collect_frames_sb3(env_show, ppo_model, n_steps=300)
print(f'Crawler traveled to x = {env_show.data.qpos[0]:.3f}')
env_show.close()
show_video(frames_ppo, title='PPO: continuous actions, stable training')

---
## Demo 3: PPO scales to complex bodies

The real power of PPO: the **same algorithm** works on much harder problems.

Let's try **Ant-v4** from Gymnasium:
- 4 legs, 8 joints
- State: 111-dimensional (joint angles, velocities, contact forces, ...)
- Action: 8-dimensional continuous torques

We change nothing about the algorithm — just point it at a harder environment.

In [ ]:
# ---------- Demo 3: PPO on Ant-v4 ----------
print('=== Demo 3: PPO on Ant-v4 ===')
print('111D state, 8D continuous action — same algorithm, no changes.\n')

env_ant = gym.make('Ant-v4', render_mode='rgb_array')
print(f'Observation space: {env_ant.observation_space.shape}')
print(f'Action space: {env_ant.action_space.shape}\n')

env_ant_train = Monitor(gym.make('Ant-v4'))
logger_ant = RewardLogger()

t0 = time.time()
ppo_ant = PPO(
    'MlpPolicy', env_ant_train,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1
)

# This takes a few minutes on Colab
ppo_ant.learn(total_timesteps=500_000, callback=logger_ant)
print(f'\nTraining completed in {time.time()-t0:.0f}s')

In [ ]:
# Training curve for Ant
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
if logger.episode_rewards:
    plot_rewards(logger.episode_rewards, title='PPO on Crawler (2D action)', ax=axes[0])
if logger_ant.episode_rewards:
    plot_rewards(logger_ant.episode_rewards, title='PPO on Ant (8D action)', ax=axes[1])
plt.tight_layout()
plt.show()

print('Same algorithm. Same hyperparameters. Much harder problem.')
print('PPO handles it — this is why it is the workhorse of motion control.')

In [ ]:
# Watch the Ant walk!
frames_ant = []
obs, _ = env_ant.reset()
for _ in range(300):
    frames_ant.append(env_ant.render())
    action, _ = ppo_ant.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = env_ant.step(action)
    if terminated or truncated:
        obs, _ = env_ant.reset()

env_ant.close()
show_video(frames_ant, title='PPO on Ant-v4: 8 joints, continuous torques')

---
## Demo 4: Reward engineering — same algorithm, different behavior

One of the most important practical "secret sauces": **the reward function shapes the behavior.**

Let's train the same crawler with different rewards:
1. **Forward velocity only** (what we've been doing)
2. **Forward velocity − energy penalty** (prefer efficient locomotion)
3. **Forward velocity − control cost** (prefer smooth, small torques)

Same algorithm, same network, same hyperparameters. Only the reward changes.

In [ ]:
# ---------- Demo 4: Reward engineering ----------
reward_configs = {
    'velocity only': dict(energy_penalty=0.0, ctrl_cost=0.0),
    'velocity - energy': dict(energy_penalty=0.3, ctrl_cost=0.0),
    'velocity - control cost': dict(energy_penalty=0.0, ctrl_cost=0.5),
}

reward_models = {}
reward_logs = {}

for name, config in reward_configs.items():
    print(f'\n=== Training with reward: {name} ===')
    env = Monitor(CrawlerGymEnv(max_steps=500, **config))
    logger_r = RewardLogger()

    model = PPO('MlpPolicy', env, learning_rate=3e-4, n_steps=2048,
                batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95,
                clip_range=0.2, verbose=0)
    model.learn(total_timesteps=80_000, callback=logger_r)

    reward_models[name] = model
    reward_logs[name] = logger_r.episode_rewards
    print(f'  Final avg reward: {np.mean(logger_r.episode_rewards[-20:]):.1f}')

In [ ]:
# Compare training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, rews) in zip(axes, reward_logs.items()):
    plot_rewards(rews, title=name, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Watch all three side by side
for name, model in reward_models.items():
    config = reward_configs[name]
    env_show = CrawlerGymEnv(max_steps=300, **config)
    frames = collect_frames_sb3(env_show, model, n_steps=200)
    x_final = env_show.data.qpos[0]
    env_show.close()
    print(f'\n{name}: traveled to x = {x_final:.3f}')
    display(show_video(frames, title=f'Reward: {name}'))

### What to notice

- **Velocity only:** The crawler moves fast but may use wild, jerky motions
- **Velocity − energy:** The crawler finds a more efficient gait (less flailing)
- **Velocity − control cost:** The crawler uses smaller, smoother torques

The algorithm didn't change. The network didn't change. Only the **reward function** changed — and the emergent behavior is completely different.

This is why reward engineering is one of the most important practical skills in RL for motion control. As the DeepMimic paper shows, combining task rewards with imitation rewards (matching a reference motion) produces natural-looking locomotion.

---
## Summary

| Challenge | Solution |
|-----------|----------|
| Q-learning can't handle continuous actions | **Policy gradient:** directly output continuous torques |
| REINFORCE has high variance, slow convergence | **PPO:** critic baseline + clipped updates + GAE |
| Same algorithm, different morphologies | PPO scales from 2-joint crawler to 8-joint Ant |
| Same algorithm, different behaviors | **Reward engineering** shapes the emergent behavior |

### The RL pipeline for motion control

1. Design state/action space
2. Design reward function (the hard part!)
3. Run PPO
4. Iterate on reward until behavior looks right

**Key insight:** In practice, more time is spent on reward design than on the RL algorithm itself.